# Flood Image Analysis — Qwen3-VL + Storm Inlet Proximity

A testable workflow that does not require complete API access up front.

1. Load image metadata and match local image files.
2. Test Qwen3-VL on one image.
3. Classify images in batches, saving a checkpoint after each image.
4. Retrieve StormInlets data or load a local GeoJSON fallback.
5. Calculate distance from positively documented flood observations to the nearest mapped inlet.
6. Export results and make a basic plot.

**Interpretation safeguards:** an image without visible flooding is not evidence that its location did not flood. Image coordinates may represent footprint centers; use footprint bounds where available. Proximity does not establish drainage capacity, condition, effectiveness, or causation. TCEQ sanitary sewer overflow data is a separate wastewater incident dataset, not storm-drain infrastructure.

Run cells from top to bottom. Start with the one-image Qwen test.

## 1. Install dependencies

In [ ]:
%pip install -q pandas requests pillow matplotlib openai geopandas shapely pyproj

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import os, re, json, time, base64
from io import BytesIO
import pandas as pd
import requests
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from openai import OpenAI

# Update these paths for your folder layout.
PROJECT_DIR = Path.cwd()
IMAGE_DIR = PROJECT_DIR / "images"
METADATA_CSV = PROJECT_DIR / "image_metadata.csv"
OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLASSIFICATION_CSV = OUTPUT_DIR / "qwen_image_classifications.csv"
STORM_GEOJSON = PROJECT_DIR / "storm_inlets.geojson"  # Optional downloaded fallback

# Your existing OpenAI-compatible Qwen server.
# Environment variables are safer than committing credentials to GitHub.
QWEN_BASE_URL = os.getenv("QWEN_BASE_URL", "http://10.246.100.230/v1")
QWEN_API_KEY = os.getenv("QWEN_API_KEY", "usaadatathon")
QWEN_MODEL = "qwen3-vl-8b-instruct"

client = OpenAI(
    base_url=QWEN_BASE_URL,
    api_key=QWEN_API_KEY,
    timeout=120.0,
    max_retries=2,
)

print("Project:", PROJECT_DIR.resolve())
print("Images folder exists:", IMAGE_DIR.exists())
print("Metadata exists:", METADATA_CSV.exists())
print("Qwen endpoint:", QWEN_BASE_URL)
print("Model:", QWEN_MODEL)

## 3. Load and inspect metadata

In [ ]:
if not METADATA_CSV.exists():
    raise FileNotFoundError(f"Metadata file not found: {METADATA_CSV}")

metadata = pd.read_csv(METADATA_CSV)
print("Rows:", len(metadata), "| Columns:", len(metadata.columns))
display(metadata.head())

for col in ["latitude", "longitude", "west", "east", "north", "south"]:
    if col in metadata.columns:
        metadata[col] = pd.to_numeric(metadata[col], errors="coerce")

expected = ["image_id", "image_path", "latitude", "longitude"]
print("Missing expected fields:", [c for c in expected if c not in metadata.columns])
print("Available columns:", metadata.columns.tolist())

## 4. Match metadata rows to local image files

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}")

image_files = [p for p in IMAGE_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
by_name, by_stem = {}, {}
for p in image_files:
    by_name.setdefault(p.name.lower(), p)
    by_stem.setdefault(p.stem.lower(), p)

def resolve_image_path(row):
    raw = row.get("image_path")
    if pd.notna(raw):
        p = Path(str(raw))
        for candidate in [p, IMAGE_DIR / p, IMAGE_DIR / p.name]:
            if candidate.exists() and candidate.is_file():
                return str(candidate.resolve())
        if p.name.lower() in by_name:
            return str(by_name[p.name.lower()].resolve())
        if p.stem.lower() in by_stem:
            return str(by_stem[p.stem.lower()].resolve())
    image_id = row.get("image_id")
    if pd.notna(image_id):
        key = str(image_id).strip().lower()
        if key in by_name:
            return str(by_name[key].resolve())
        if key in by_stem:
            return str(by_stem[key].resolve())
        for ext in IMAGE_EXTENSIONS:
            if key + ext in by_name:
                return str(by_name[key + ext].resolve())
    return None

metadata["resolved_image_path"] = metadata.apply(resolve_image_path, axis=1)
matched = metadata["resolved_image_path"].notna()
print("Image files found:", len(image_files))
print("Metadata records matched:", int(matched.sum()))
print("Metadata records unmatched:", int((~matched).sum()))
print("Approx. image files without metadata:", max(0, len(image_files) - int(matched.sum())))
display(metadata.loc[~matched, [c for c in ["image_id", "image_path"] if c in metadata.columns]].head(20))
metadata.to_csv(OUTPUT_DIR / "metadata_with_resolved_paths.csv", index=False)

## 5. Inspect sample images

In [ ]:
sample = metadata.loc[metadata["resolved_image_path"].notna()].head(5)
for _, row in sample.iterrows():
    print("ID:", row.get("image_id"), "|", row["resolved_image_path"])
    try:
        display(Image.open(row["resolved_image_path"]).convert("RGB"))
    except Exception as e:
        print("Image read error:", repr(e))

## 6. Define Qwen classification

In [ ]:
CLASSIFICATION_PROMPT = '''
Classify only visible evidence in this image for a study of documented flood evidence
and proximity to mapped storm-drain infrastructure.

Return ONLY one valid JSON object with these keys:
{
  "visible_flooding": "yes | no | uncertain",
  "flood_aftermath": "yes | no | uncertain",
  "visible_drainage_overflow": "yes | no | uncertain",
  "visible_drainage_infrastructure": "yes | no | uncertain",
  "water_on_road_or_pavement": "yes | no | uncertain",
  "evidence_description": "brief description of visible evidence",
  "confidence": "high | medium | low",
  "limitations": "what cannot be determined from this image"
}

Rules:
- Do not infer flooding solely because a surface is wet.
- Do not infer a drain is blocked, undersized, damaged, or ineffective unless directly visible.
- Do not infer location from visual content.
- "Not visible" is not the same as "not present". Use uncertain if the image is insufficient.
- Describe observations, not causes.
'''

def image_to_data_url(image_path, max_side=1400, jpeg_quality=88):
    with Image.open(image_path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_side, max_side))
        buffer = BytesIO()
        im.save(buffer, format="JPEG", quality=jpeg_quality)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("utf-8")

def parse_json_response(content):
    text = (content or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise

def classify_one_image(image_path):
    data_url = image_to_data_url(image_path)
    response = client.chat.completions.create(
        model=QWEN_MODEL,
        messages=[
            {"role": "system", "content": "You are a careful visual evidence classifier. Return the requested JSON."},
            {"role": "user", "content": [
                {"type": "text", "text": CLASSIFICATION_PROMPT},
                {"type": "image_url", "image_url": {"url": data_url}},
            ]},
        ],
        temperature=0.1,
        max_tokens=500,
    )
    raw = response.choices[0].message.content
    return parse_json_response(raw), raw

## 7. Test one image before processing the collection

In [ ]:
test_rows = metadata.loc[metadata["resolved_image_path"].notna()]
if test_rows.empty:
    raise ValueError("No image paths resolved. Fix the image matching step first.")

test_row = test_rows.iloc[0]
print("Testing image:", test_row.get("image_id"), test_row["resolved_image_path"])
test_result, test_raw = classify_one_image(test_row["resolved_image_path"])
display(pd.DataFrame([test_result]))
print("Raw response:\n", test_raw)

## 8. Batch classification with resumable checkpoints

Start with `MAX_IMAGES = 5`. After confirming output quality, set it to `None` to process all remaining images. Existing image IDs in the output CSV are skipped.

In [ ]:
MAX_IMAGES = 5
PAUSE_SECONDS = 0.2

if CLASSIFICATION_CSV.exists():
    existing = pd.read_csv(CLASSIFICATION_CSV)
    completed_ids = set(existing["image_id"].astype(str))
    results = existing.to_dict("records")
else:
    completed_ids, results = set(), []

todo = metadata.loc[metadata["resolved_image_path"].notna()].copy()
todo = todo[~todo["image_id"].astype(str).isin(completed_ids)]
if MAX_IMAGES is not None:
    todo = todo.head(MAX_IMAGES)

print("Images queued this run:", len(todo))
for i, (_, row) in enumerate(todo.iterrows(), start=1):
    record = {
        "image_id": str(row.get("image_id")),
        "resolved_image_path": row["resolved_image_path"],
    }
    for col in ["latitude", "longitude", "west", "east", "north", "south",
                "image_role", "image_view_type", "event_name", "pair_id",
                "paired_image_id", "candidate_condition"]:
        if col in row.index:
            record[col] = row[col]
    try:
        labels, raw = classify_one_image(row["resolved_image_path"])
        record.update(labels)
        record["analysis_status"] = "success"
        record["error"] = None
    except Exception as e:
        record["analysis_status"] = "error"
        record["error"] = repr(e)
        print("Error on", record["image_id"], ":", repr(e))

    results.append(record)
    pd.DataFrame(results).to_csv(CLASSIFICATION_CSV, index=False)
    print(f"{i}/{len(todo)} — {record['image_id']} — {record['analysis_status']}")
    time.sleep(PAUSE_SECONDS)

print("Checkpoint saved to:", CLASSIFICATION_CSV)
display(pd.DataFrame(results).tail())

## 9. Review model outputs

In [ ]:
if CLASSIFICATION_CSV.exists():
    classifications = pd.read_csv(CLASSIFICATION_CSV)
    print("Classification rows:", len(classifications))
    if "analysis_status" in classifications:
        display(classifications["analysis_status"].value_counts(dropna=False))
    if "visible_flooding" in classifications:
        display(classifications["visible_flooding"].value_counts(dropna=False))
    display(classifications.head(10))
else:
    print("Run the Qwen test and batch cells first.")

## 10. Test the StormInlets ArcGIS endpoint

In [ ]:
STORM_QUERY_URL = (
    "https://services.arcgis.com/g1fRTDLeMgspWrYp/"
    "arcgis/rest/services/StormInlets/FeatureServer/0/query"
)

def inspect_storm_api():
    params = {
        "where": "1=1", "outFields": "*", "returnGeometry": "true",
        "resultRecordCount": 5, "f": "geojson"
    }
    try:
        response = requests.get(STORM_QUERY_URL, params=params, timeout=45)
        print("HTTP:", response.status_code)
        print("Content type:", response.headers.get("content-type"))
        print("Response preview:\n", response.text[:2500])
        response.raise_for_status()
        payload = response.json()
        print("Top-level keys:", list(payload.keys()))
        print("Sample feature count:", len(payload.get("features", [])))
        if payload.get("features"):
            display(pd.Series(payload["features"][0].get("properties", {})))
        return payload
    except Exception as e:
        print("StormInlets request failed:", repr(e))
        return None

storm_sample = inspect_storm_api()

## 11. Download StormInlets features with pagination

Run this if the sample request succeeds. If it fails, download the layer as GeoJSON separately and place it at `storm_inlets.geojson` beside the notebook.

In [ ]:
def download_storm_inlets(page_size=2000, max_pages=100):
    all_features = []
    offset = 0
    for page in range(max_pages):
        params = {
            "where": "1=1", "outFields": "*", "returnGeometry": "true",
            "resultOffset": offset, "resultRecordCount": page_size, "f": "geojson"
        }
        response = requests.get(STORM_QUERY_URL, params=params, timeout=90)
        response.raise_for_status()
        payload = response.json()
        if "error" in payload:
            raise RuntimeError(payload["error"])
        features = payload.get("features", [])
        all_features.extend(features)
        print(f"Page {page + 1}: {len(features)} features; total {len(all_features)}")
        if len(features) < page_size:
            break
        offset += len(features)
    return {"type": "FeatureCollection", "features": all_features}

storm_geojson = None
try:
    storm_geojson = download_storm_inlets()
    if storm_geojson["features"]:
        with open(OUTPUT_DIR / "storm_inlets.geojson", "w", encoding="utf-8") as f:
            json.dump(storm_geojson, f)
        print("Saved", len(storm_geojson["features"]), "features")
    else:
        print("No features returned.")
except Exception as e:
    print("Download failed:", repr(e))

## 12. Load storm inlet GeoJSON

In [ ]:
import geopandas as gpd

if storm_geojson is not None and storm_geojson.get("features"):
    storm_gdf = gpd.GeoDataFrame.from_features(storm_geojson["features"], crs="EPSG:4326")
elif STORM_GEOJSON.exists():
    storm_gdf = gpd.read_file(STORM_GEOJSON)
else:
    storm_gdf = None
    print("No StormInlets data loaded. Spatial analysis can be run later.")

if storm_gdf is not None:
    print("Features:", len(storm_gdf))
    print("CRS:", storm_gdf.crs)
    print("Geometry types:", storm_gdf.geometry.geom_type.value_counts().to_dict())
    display(storm_gdf.head())

## 13. Calculate nearest-inlet distance for positive flood observations

Uses image footprint polygons when all four bounds are available; otherwise uses the metadata coordinate point. Distances are measured in a local UTM projection. Images labelled `no` or `uncertain` are not used as non-flooded controls.

In [ ]:
from shapely.geometry import Point, box

def observation_geometry(row):
    bounds = ["west", "south", "east", "north"]
    if all(c in row.index and pd.notna(row[c]) for c in bounds):
        west, south, east, north = [float(row[c]) for c in bounds]
        if west < east and south < north:
            return box(west, south, east, north)
    if pd.notna(row.get("longitude")) and pd.notna(row.get("latitude")):
        return Point(float(row["longitude"]), float(row["latitude"]))
    return None

distance_output = OUTPUT_DIR / "flood_observation_inlet_distances.csv"

if storm_gdf is None:
    print("Load storm inlet data first.")
elif not CLASSIFICATION_CSV.exists():
    print("Run image classification first.")
else:
    classifications = pd.read_csv(CLASSIFICATION_CSV)
    if "visible_flooding" not in classifications.columns:
        raise ValueError("Classification output has no visible_flooding field.")

    flood_obs = classifications[
        classifications["visible_flooding"].astype(str).str.lower().eq("yes")
    ].copy()

    meta_cols = [c for c in ["image_id", "latitude", "longitude", "west", "east", "north", "south"]
                 if c in metadata.columns]
    flood_obs = flood_obs.merge(
        metadata[meta_cols].drop_duplicates("image_id"),
        on="image_id", how="left", suffixes=("", "_metadata")
    )
    for col in ["latitude", "longitude", "west", "east", "north", "south"]:
        other = col + "_metadata"
        if other in flood_obs.columns:
            if col not in flood_obs.columns:
                flood_obs[col] = flood_obs[other]
            else:
                flood_obs[col] = flood_obs[col].combine_first(flood_obs[other])

    flood_obs["geometry"] = flood_obs.apply(observation_geometry, axis=1)
    flood_obs = flood_obs[flood_obs["geometry"].notna()].copy()

    if flood_obs.empty:
        print("No positive flood classifications with usable coordinates.")
    else:
        flood_gdf = gpd.GeoDataFrame(flood_obs, geometry="geometry", crs="EPSG:4326")
        if storm_gdf.crs is None:
            raise ValueError("Storm inlet data has no CRS. Set its verified CRS before calculating distances.")
        center = flood_gdf.geometry.unary_union.centroid
        utm_zone = int((center.x + 180) // 6) + 1
        utm_epsg = (32600 if center.y >= 0 else 32700) + utm_zone
        flood_m = flood_gdf.to_crs(epsg=utm_epsg)
        storm_m = storm_gdf.to_crs(epsg=utm_epsg)
        storm_m = storm_m[storm_m.geometry.notna()]
        if storm_m.empty:
            raise ValueError("Storm inlet layer contains no valid geometries.")

        nearest_m, nearest_index = [], []
        for geom in flood_m.geometry:
            d = storm_m.geometry.distance(geom)
            idx = d.idxmin()
            nearest_m.append(float(d.loc[idx]))
            nearest_index.append(str(idx))

        flood_m["nearest_inlet_distance_m"] = nearest_m
        flood_m["nearest_inlet_feature_index"] = nearest_index
        result = pd.DataFrame(flood_m.drop(columns="geometry"))
        result.to_csv(distance_output, index=False)
        print("Saved spatial results:", distance_output)
        display(result[["image_id", "nearest_inlet_distance_m"]].head(20))

## 14. Summarize nearest-inlet distances

In [ ]:
if distance_output.exists():
    spatial_results = pd.read_csv(distance_output)
    distances = pd.to_numeric(spatial_results["nearest_inlet_distance_m"], errors="coerce").dropna()
    display(distances.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).to_frame("distance_m"))
    plt.figure(figsize=(8, 4))
    plt.hist(distances, bins=25)
    plt.xlabel("Distance to nearest mapped storm inlet (meters)")
    plt.ylabel("Documented flood observations")
    plt.title("Nearest mapped storm inlet distance")
    plt.tight_layout()
    plt.show()
else:
    print("No spatial output yet.")

## 15. Optional: inspect the TCEQ sanitary sewer overflow API

This is a supplementary wastewater incident dataset, not a storm inlet inventory. Start with a small response to avoid the earlier oversized response.

In [ ]:
TCEQ_URL = "https://data.texas.gov/api/v3/views/8kc5-95uk/query.json"
try:
    response = requests.get(TCEQ_URL, params={"$limit": 5}, timeout=45)
    print("HTTP:", response.status_code)
    print("Content type:", response.headers.get("content-type"))
    print(response.text[:3000])
except Exception as e:
    print("TCEQ request failed:", repr(e))

## 16. Final validation checklist

Before interpreting results:
- Manually review a sample of Qwen labels, especially uncertain cases.
- Verify the StormInlets CRS and geometry type before distance calculations.
- Verify that footprint bounds (`west`, `east`, `north`, `south`) are longitude/latitude.
- Reconcile image file count with metadata row count and investigate unmatched files.
- Preserve before/response pairing in any comparisons.
- Do not use images labelled `no` or `uncertain` as non-flooded control locations.

Outputs are saved under `results/`: resolved metadata paths, Qwen classifications, downloaded StormInlets GeoJSON, and nearest-inlet distances.